In [ ]:
import pandas as pd
# Load dates, companies, prices, and variances
dates = pd.read_csv("Dataset/raw/Capire_March2025_Dates.csv", header=None, names=["Date"])
dates["Date"] = pd.to_datetime(dates["Date"])
companies = pd.read_csv("Dataset/raw/Capire_March2025_Companies.csv", header=None, names=["Company"])
prices = pd.read_csv("Dataset/raw/Capire_March2025_EoD_Price_1min.csv", header=None)
variances = pd.read_csv("Dataset/raw/Capire_March2025_RealizedVariance_1min.csv", header=None)
# Assign column names and insert Date column
prices.columns = companies["Company"]
prices.insert(0, "Date", dates["Date"])
variances.columns = companies["Company"]
variances.insert(0, "Date", dates["Date"])
# Save to Excel with two sheets
with pd.ExcelWriter("Dataset/dataset.xlsx") as writer:
    prices.to_excel(writer, sheet_name="EoD_Prices", index=False)
    variances.to_excel(writer, sheet_name="Realized_Variances", index=False)


In [3]:
import numpy as np
import pandas as pd

# --- file paths ---
xlsx_path = "Dataset/dataset.xlsx"
csv_path  = "Dataset/raw/FF5_daily.csv"

# --- date bounds ---
first_date = pd.Timestamp("2003-01-02")
last_date  = pd.Timestamp("2025-03-31")

# --- read raw CSV ---
ff5_raw = pd.read_csv(csv_path)

# --- parse date column ---
# first column is YYYYMMDD, keep name as "DATE"
ff5_raw = ff5_raw.rename(columns={ff5_raw.columns[0]: "DATE"})
ff5_raw["DATE"] = pd.to_datetime(ff5_raw["DATE"], format="%Y%m%d")

# --- restrict date range ---
ff5_raw = ff5_raw.loc[
    (ff5_raw["DATE"] >= first_date) &
    (ff5_raw["DATE"] <= last_date)
].reset_index(drop=True)

# --- handle missing values ---
# -99.99 is a missing-value sentinel
ff5_raw = ff5_raw.replace(-99.99, np.nan)

# interpolate using arithmetic mean of adjacent values
ff5_filled = ff5_raw.copy()
ff5_filled.iloc[:, 1:] = (
    ff5_filled
    .iloc[:, 1:]
    .interpolate(method="linear", limit_direction="both")
)

# --- sanity check ---
if ff5_filled.iloc[:, 1:].isna().any().any():
    raise ValueError("Interpolation left unresolved missing values.")

# --- write to Excel as sheet 'FF5' ---
with pd.ExcelWriter(
    xlsx_path,
    mode="a",
    if_sheet_exists="replace"
) as writer:
    ff5_filled.to_excel(writer, sheet_name="FF5", index=False)
